<a href="https://colab.research.google.com/github/Fatima-Eman-hub/fatima-eman-flyrank-ml-01/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*


**1. Introduction & Problem Statement**


FlyRank's content teams are responsible for reviewing large inventories of client pages with limited time. In practice, this means picking a handful of pages to look at first out of hundreds or thousands. The existing approach relies on hand-written rules—for example, flagging a page as worth reviewing if it is both stale (not updated in 180+ days) and still visible (500+ impressions). This works as a starting point, but it is a coarse filter: it can only combine a couple of signals with a hard cutoff, and it cannot weigh position, traffic, and freshness together the way a learned model can.

The decision this work supports is simple to state and hard to get right:

> **Given everything knowable about a page today, which pages are worth a human's limited review time first?**

Getting this wrong has a real, if modest, cost—time spent reviewing a page that did not need attention while genuinely declining pages wait longer for review.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

This work uses the **FlyRank ML Internship warehouse release (build v20260703)**, a pseudonymized snapshot of real client search-performance data hosted on Hugging Face.

## Tables Used

| Table | Description |
|--------|-------------|
| `fact_content_daily_performance` | 78,835,655 rows at the grain of report_date × client × content item (2025-01-27 → 2026-06-30). |
| `dim_content` | 519,606 rows containing one record per pseudonymized content item with content creation dates. |

## Feature Window

Features were constructed from **March 2026**, deliberately chosen as a mid-panel month rather than the final month of the release.

## Prediction Label

The target label was constructed using **April 2026** impressions so that every feature is observable strictly before the outcome it predicts.

## Excluded Features

The following FlyRank product decision fields were intentionally excluded:

- `health_score`
- `priority_score`
- `action_type`
- `refresh_tier`

These were omitted to prevent the model from simply reproducing existing product decisions instead of learning new predictive signals.

Additionally, raw client names, domains, URLs, and search queries were never used anywhere in this work.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Assumption

The central assumption is that a page's near-term traffic trajectory (declining vs. not declining) can be predicted using only observable historical signals, and that these relationships generalize across approximately 70 different clients.

---

## Features

| Feature | Why it is available at decision time |
|---------|---------------------------------------|
| `impressions_march` | Google Search Console impressions are fully known by the end of March. |
| `avg_position` | Search position is recorded daily and completely observable before prediction. |
| `content_age_days` | Computed from the historical publication date. |
| `days_with_ga4` | Counts only already-observed analytics days. |

---

## Prediction Label

`is_declining = 1`

if April 2026 impressions are lower than March 2026 impressions, otherwise `0`.

---

## Baseline

A simple rule-based baseline flags pages satisfying:

- `content_age_days >= 180`
- `impressions_march >= 500`

Qualified pages are ranked only by traffic volume.

---

## Models

Two supervised learning models were trained:

- Logistic Regression
- Random Forest

Both models used:

- `class_weight="balanced"`
- `random_state = 42`

---

## Validation Strategy

A **GroupShuffleSplit (75/25)** grouped by client ensures pages from the same client never appear in both training and testing datasets.

This avoids client-specific leakage and evaluates how well the model generalizes to unseen clients.

---

## Leakage Checks

The following safeguards were implemented:

- Product decision flags were excluded.
- Label-derived fields (`trend_direction`, `trend_pct`) were excluded.
- Preprocessing was fit only on the training data.
- A comparison against a naive random split was performed to quantify leakage.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## Precision@50 Comparison

| Method | Precision@50 |
|---------|-------------:|
| Baseline Rule (stale + visible) | 0.500 |
| Base Rate (random guessing) | 0.578 |
| Logistic Regression | 0.660 |
| **Random Forest** | **0.700** |

The Random Forest achieved the highest Precision@50 on the client-grouped validation split, outperforming both the rule-based baseline and logistic regression.

An important finding is that the same Random Forest reached **0.900 Precision@50** under a naive random train/test split, compared with **0.700** using the client-grouped split. This large performance gap demonstrates the effect of client leakage and validates the importance of grouped evaluation.

Feature importance from the Random Forest ranked the predictors as follows:

| Feature | Importance |
|---------|-----------:|
| `avg_position` | 0.366 |
| `content_age_days` | 0.287 |
| `impressions_march` | 0.284 |
| `days_with_ga4` | 0.063 |

## 5. Limitations

*What this work cannot claim.*

- This is a **decision-support system**, not a causal model. High-risk pages are candidates for manual review rather than evidence that refreshing them will improve traffic.
- Results are based on a single month-to-month prediction window (March → April 2026), so seasonal effects remain untested.
- Client history depth varies substantially across the dataset, making generalization uneven for clients with limited historical data.
- The baseline scoring rule ranks qualifying pages only by traffic volume and ignores search position, while the learned model only partially addresses this limitation.
- `content_age_days`, although highly important, may capture patterns specific to this training split. Several development false positives suggest caution before treating its contribution as universally stable.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

| Reason Code | Suggested Action | Trigger |
|--------------|-----------------|---------|
| `stale_high_value` | Review for refresh | Age ≥ 365 days and impressions ≥ 500 |
| `aging_visible_page` | Review for refresh | Age ≥ 180 days and impressions ≥ 500 |
| `visible_low_rank` | Review for CTR / intent improvements | Impressions ≥ 500 and average position > 10 |

These recommendations are intended only as decision-support for human reviewers.

Before taking any action, reviewers should verify:

- Whether the page has already been updated recently.
- Whether traffic has shifted to another related page.
- Whether the suggested reason code matches the page's actual condition.

The system should **never** automatically refresh or publish content based solely on the model's predictions, and FlyRank's internal product decision fields should never be fed back into the model as training features.

Monitoring should focus on changes in client distribution, reductions in Precision@50 relative to the 0.700 benchmark, and shifts in the number of pages meeting the visibility/staleness thresholds.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

**Artifacts embedded in this paper**

The paper embeds two primary artifacts generated during model development:

1. **Precision@50 comparison table** (Results section)
2. **Random Forest feature importance values** (Methodology/Results)

Both artifacts are sourced directly from the outputs of:

- `w05_model.ipynb`
- `w06_validation_audit.ipynb`

These artifacts provide quantitative evidence supporting the paper's evaluation and model interpretation.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.